## Enable GPU Acceleration

---
Before you start exploring this notebook make sure that GPU support is enabled.
To enable the GPU backend for your notebook, go to **Edit** â†’ **Notebook Settings** and set **Hardware accelerator** to **GPU**.

---


# Set Up

In [ ]:
import os
import sys
import platform
import subprocess
from pathlib import Path

# User settings
USE_GOOGLE_DRIVE = True
VISUALIZE = True

# Detect operating system
OS_NAME = platform.system()

# Detect whether the notebook is running inside Google Colab
try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Print system information
print(f"Detected system: {OS_NAME}")
print(f"Running in Colab: {IN_COLAB}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Current working directory: {Path.cwd()}")

# Helper function for running terminal commands
def run_cmd(cmd, check=True):
    subprocess.run(cmd, check=check)

# Helper function for installing Python packages
def pip_install(packages):
    if isinstance(packages, str):
        packages = [packages]
    run_cmd([sys.executable, "-m", "pip", "install", *packages])

# Install Linux system packages only when running in Colab
if IN_COLAB:
    run_cmd(["apt", "update"])
    run_cmd([
        "apt", "install", "-y",
        "swig",
        "python3-numpy",
        "python3-dev",
        "cmake",
        "zlib1g-dev",
        "libjpeg-dev",
        "xvfb",
        "ffmpeg",
        "xorg-dev",
        "python3-opengl",
        "libboost-all-dev",
        "libsdl2-dev",
    ])

# Install required Python packages
pip_install([
    "gymnasium==1.2.3",
    "gymnasium[box2d]",
    "imageio-ffmpeg",
    "moviepy==1.0.3",
    "onnx==1.21.0",
    "onnx2pytorch==0.5.3",
    "onnxscript==0.7.0",
    "numpy==2.4.2",
    "opencv-python-headless",
])

# Install virtual display support only on Linux or Colab
if IN_COLAB or OS_NAME == "Linux":
    pip_install(["pyvirtualdisplay"])

# Mount Google Drive only inside Colab
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive")
else:
    BASE_DIR = Path.cwd()

# Select CUDA GPU 0 if CUDA is available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Detect whether PyTorch can use GPU or CPU
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ModuleNotFoundError:
    DEVICE = "cpu"

# Print final paths and device information
print(f"Base directory: {BASE_DIR}")
print(f"Device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

# Imports

Install Gymnasium and dependencies to render the environments

In [ ]:
%matplotlib inline

# Auxiliary Python imports
import math
import io
import base64
import random
import shutil
from time import time, strftime
from glob import glob
from tqdm import tqdm
import numpy as np

# Pytorch
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical

# Cross-framework library for DL
import onnx
# import onnxruntime
import onnx2pytorch
from onnx2pytorch import ConvertModel

# Environment import
import gymnasium as gym
from gymnasium.spaces import Box
from gymnasium.wrappers import RecordVideo
import logging
logging.getLogger("gymnasium").setLevel(logging.ERROR)

# Plotting and notebook imports
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, clear_output
from IPython import display

print("Python:", sys.version)
print("torch:", torch.__version__)
print("gymnasium:", gym.__version__)
print("numpy:", np.__version__)
print("onnx:", onnx.__version__)
print("onnx2pytorch:", onnx2pytorch.__version__)

# -------- Virtual display (Linux / Colab only) --------
pydisplay = None

if OS_NAME == "Linux" and VISUALIZE:
    try:
        from pyvirtualdisplay import Display
        pydisplay = Display(visible=0, size=(640, 480))
        pydisplay.start()
        print("Virtual display started.")
    except Exception as e:
        print("Skipping virtual display:", e)
else:
    print(f"Skipping virtual display on {OS_NAME} (not needed).")


# Select device for training

By default we train on GPU if one is available, otherwise we fall back to the CPU.
If you want to always use the CPU change accordingly.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: " + str(device))

# Download Dataset and Expert model

In [ ]:
# Import gdown, and install it first if it is missing
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

# Use the current notebook folder as the base directory
BASE_DIR = os.getcwd()

# Download one file from Google Drive if it does not already exist
def download_file(file_id, output_name):
    output_path = os.path.join(BASE_DIR, output_name)

    if os.path.isfile(output_path):
        print(f"{output_name} already exists, skipping download.")
        return output_path

    gdown.download(id=file_id, output=output_path, quiet=False)
    return output_path

# Download a zip file and extract it if the expected folder does not already exist
def download_and_extract(file_id, zip_name, expected_dir):
    expected_path = os.path.join(BASE_DIR, expected_dir)

    if os.path.isdir(expected_path):
        print(f"{expected_dir} already exists, skipping download and extraction.")
        return

    zip_path = download_file(file_id, zip_name)

    print(f"Extracting {zip_name}...")
    shutil.unpack_archive(zip_path, BASE_DIR, "zip")
    print(f"Extracted {zip_name}")

# Download the expert ONNX model
download_file(
    "1h85_VOCN9mEne9ZSx4TZA5hbluXDh-Be",
    "expert.onnx"
)

# Download and extract the training dataset
download_and_extract(
    "1mWVhLk6uTlyc66OBkBaKCGPQpkzdGzqR",
    "train.zip",
    "train"
)

# Download and extract the validation dataset
download_and_extract(
    "1T0Zny_ZIMTJYRmDUgxhWSq1-na_ccW4q",
    "val.zip",
    "val"
)

# Check that all required files and folders exist
print("Current directory:", BASE_DIR)
print("expert.onnx exists:", os.path.isfile(os.path.join(BASE_DIR, "expert.onnx")))
print("train exists:", os.path.isdir(os.path.join(BASE_DIR, "train")))
print("val exists:", os.path.isdir(os.path.join(BASE_DIR, "val")))

# Auxiliary Methods

The following cell contains classes and functions to provide some functionality for logging, plotting and exporting your model in the format required by the submission server.
You are free to use your own logging framework if you wish (such as tensorboard or Weights & Biases).
The logger is a very simple implementation of a CSV-file based logger.
Additionally it creates a folder for each run with subfolders for model files, logs and videos.

In [ ]:
class Logger():
    def __init__(self, logdir, params=None):
        self.basepath = os.path.join(logdir, strftime("%Y-%m-%dT%H-%M-%S"))
        os.makedirs(self.basepath, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)
        if params is not None and os.path.exists(params):
            shutil.copyfile(params, os.path.join(self.basepath, "params.pkl"))
        self.log_dict = {}
        self.dump_idx = {}

    @property
    def param_file(self):
        return os.path.join(self.basepath, "params.pkl")

    @property
    def onnx_file(self):
        return os.path.join(self.basepath, "model.onnx")

    @property
    def video_dir(self):
        return os.path.join(self.basepath, "videos")

    @property
    def log_dir(self):
        return os.path.join(self.basepath, "logs")

    def log(self, name, value):
        if name not in self.log_dict:
            self.log_dict[name] = []
            self.dump_idx[name] = -1
        self.log_dict[name].append((len(self.log_dict[name]), time(), value))

    def get_values(self, name):
        if name in self.log_dict:
            return [x[2] for x in self.log_dict[name]]
        return None

    def dump(self):
        for name, rows in self.log_dict.items():
            with open(os.path.join(self.log_dir, name + ".log"), "a") as f:
                for i, row in enumerate(rows):
                    if i > self.dump_idx[name]:
                        f.write(",".join([str(x) for x in row]) + "\n")
                        self.dump_idx[name] = i


def plot_metrics(logger):
    train_loss  = logger.get_values("training_loss")
    train_entropy  = logger.get_values("training_entropy")
    val_loss = logger.get_values("validation_loss")
    val_acc = logger.get_values("validation_accuracy")

    fig = plt.figure(figsize=(15,5))
    ax1 = fig.add_subplot(131, label="train")
    ax2 = fig.add_subplot(131, label="val",frame_on=False)
    ax4 = fig.add_subplot(132, label="entropy")
    ax3 = fig.add_subplot(133, label="acc")

    ax1.plot(train_loss, color="C0")
    ax1.set_ylabel("Loss")
    ax1.set_xlabel("Update (Training)", color="C0")
    ax1.xaxis.grid(False)
    ax1.set_ylim((0,4))

    ax2.plot(val_loss, color="C1")
    ax2.xaxis.tick_top()
    ax2.yaxis.tick_right()
    ax2.set_xlabel('Epoch (Validation)', color="C1")
    ax2.xaxis.set_label_position('top')
    ax2.xaxis.grid(False)
    ax2.get_yaxis().set_visible(False)
    ax2.set_ylim((0,4))

    ax4.plot(train_entropy, color="C3")
    ax4.set_xlabel('Update (Training)', color="black")
    ax4.set_ylabel("Entropy", color="C3")
    ax4.tick_params(axis='x', colors="black")
    ax4.tick_params(axis='y', colors="black")
    ax4.xaxis.grid(False)

    ax3.plot(val_acc, color="C2")
    ax3.set_xlabel("Epoch (Validation)", color="black")
    ax3.set_ylabel("Accuracy", color="C2")
    ax3.tick_params(axis='x', colors="black")
    ax3.tick_params(axis='y', colors="black")
    ax3.xaxis.grid(False)
    ax3.set_ylim((0,1))

    fig.tight_layout(pad=2.0)
    plt.show()

"""
Utility functions to enable video recording of gym environment and displaying it
"""
def show_video(video_dir):
    mp4list = glob(f'{video_dir}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data='''<video alt="test" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                 </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

def save_as_onnx(torch_model, sample_input, model_path):
    torch.onnx.export(
                    torch_model,               # model being run
                    sample_input,              # model input (or a tuple for multiple inputs)
                    f=model_path,              # where to save the model (can be a file or file-like object)
                    export_params=True,        # store the trained parameter weights inside the model file
                    opset_version=17,          # the ONNX version to export the model to - see https://github.com/microsoft/onnxruntime/blob/master/docs/Versioning.md
                    do_constant_folding=True,  # preserve architectural info (kernel sizes, strides, etc.) as explicit graph attributes
                    external_data=False,       # keeps everything in a single .onnx file
                    dynamo=False,              # export with old logic
                    )
    # Test conversion
    converted = ConvertModel(onnx.load(model_path))
    print("conversion ok")


# Dataset

Use this dataset class to load the provided demonstrations. Furthermore, this dataset has functionality to add new samples to the dataset which you will need for implementing the DAgger algorithm.

In [ ]:
class DemonstrationDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.files = sorted(glob(f"{data_dir}/*.npz"))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        state = data["state"][np.newaxis, ...].astype(np.float32)
        action = data["action"]
        return state / 255.0, action.item()

    def append(self, states, actions):
        offset = len(self) + 1
        for i in range(len(states)):
            filename = f"{self.data_dir}/{offset+i:06}.npz"
            np.savez_compressed(filename, state=states[i], action=actions[i].astype(np.int32))
            self.files.append(filename)

# Inspect data

It is always a good idea to take a look at the data when you start working with a new dataset. Feel free to investigate the dataset further on your own.

In [ ]:
# Action Statistics
dataset = DemonstrationDataset("train")
print("Number of samples: {}".format(len(dataset)));

In [ ]:
# Action mapping from gymnasium.farama.org
action_mapping = {
    0: "do nothing",
    1: "steer left",
    2: "steer right",
    3: "gas",
    4: "brake"
}

# Visualize random frames
idx = np.random.randint(len(dataset))
state, action = dataset[idx]
# store a single frame as we need it later for exporting an ONNX model (it needs a sample of the input for the export)
sample_state = torch.Tensor(state).unsqueeze(0).to(device)
# Display the sample
print(f"Action: {action_mapping[action]}")
plt.axis("off")
plt.imshow(state[0]);

In [ ]:
# release memory
del dataset

# Define Policy Network

You need to design a neural network architecture that is capable of mapping a state to an action.
The input is a single image with the following properties:
- Resolution of 84x84 pixels
- Grayscale (meaning a single channel as opposed to three channels of an RGB image)
- The values of each pixel should be between 0 and 1

The output of the network should be one unit per possible action, as our environment has 5 actions that results in 5 output units.
Your network must implement the forward function in order to be compatible with the evaluation script.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, n_units_out):
        super(PolicyNetwork, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=8, stride=4)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.relu2 = nn.ReLU()
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.relu3 = nn.ReLU()

        # Calculate the output size of the convolutional layers
        # Input: (1, 84, 84)
        # After conv1: (32, (84-8)/4 + 1, (84-8)/4 + 1) = (32, 20, 20)
        # After conv2: (64, (20-4)/2 + 1, (20-4)/2 + 1) = (64, 9, 9)
        # After conv3: (64, (9-3)/1 + 1, (9-3)/1 + 1) = (64, 7, 7)
        self.fc = nn.Linear(64 * 7 * 7, n_units_out)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = x.view(x.size(0), -1) # Flatten the output
        return self.fc(x)

# Train behavioral cloning policy

Now that you have a Dataset and a network you need to train your network.
With behavioral cloning we want to imitate the behavior of the agent that produced the demonstration dataset as close as possible.
This is basically supervised learning, where you want to minimize the loss of your network on the training and validation sets.

Some tips as to what you need to implement:
- choose the appropriate loss function (think on which kind of problem you are solving)
- choose an optimizer and its hyper-parameters
- optional: use a learning-rate scheduler
- don't forget to evaluate your network on the validation set
- store your model and training progress often so you don't loose progress if your program crashes

In case you use the provided Logger:
- `logger.log("training_loss", <loss-value>)` to log a particular value
- `logger.dump()` to write the current logs to a log file (e.g. after every episode)
- `logger.log_dir`, `logger.param_file`, `logger.onnx_file`, `logger.video_dir` point to files or directories you can use to save files
- you might want to specify your google drive folder as a logdir in order to automatically sync your results
- if you log the metrics specified in the `plot_metrics` function you can use it to visualize your training progress (or take it as a template to plot your own metrics)

In [ ]:
# choose the batchsize for training
batch_size = 64

# Datasets
train_set = DemonstrationDataset("train")
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, num_workers=2, shuffle=True, drop_last=False, pin_memory=True)
val_set = DemonstrationDataset("val")
val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, num_workers=2, shuffle=False, drop_last=False, pin_memory=True)

# Specify the google drive mount here if you want to store logs and weights there (and set it up earlier)
# You can also choose to use a different logging framework such as tensorboard (not recommended on Colab) or Weights & Biases (highly recommended)
logger = Logger("logdir")
print("Saving state to {}".format(logger.basepath))

# Network
model = PolicyNetwork(n_units_out=5)
model = model.to(device)
num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable Parameters: {}".format(num_trainable_params))

######################
### YOUR CODE HERE ###
######################
# Implement your training and evaluation loop
# feel free to define your own functions for training and evaluation

# If you want to export your model as an ONNX file use the following code as template
# If you use the provided logger you can use this directly
save_as_onnx(model, sample_state, logger.onnx_file)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20 # You can adjust the number of epochs
best_val_accuracy = -float('inf')

print("Starting Behavioral Cloning training...")
for epoch in range(num_epochs):
    # Training loop
    model.train()
    train_loss_sum = 0
    for batch_idx, (states, actions) in enumerate(train_loader):
        states, actions = states.to(device), actions.to(device)

        optimizer.zero_grad()
        logits = model(states)
        loss = criterion(logits, actions)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        logger.log("training_loss", loss.item())

    avg_train_loss = train_loss_sum / len(train_loader)

    # Validation loop
    model.eval()
    val_loss_sum = 0
    correct_predictions = 0
    total_predictions = 0
    with torch.no_grad():
        for states, actions in val_loader:
            states, actions = states.to(device), actions.to(device)
            logits = model(states)
            loss = criterion(logits, actions)
            val_loss_sum += loss.item()

            _, predicted = torch.max(logits, 1)
            total_predictions += actions.size(0)
            correct_predictions += (predicted == actions).sum().item()

    avg_val_loss = val_loss_sum / len(val_loader)
    val_accuracy = correct_predictions / total_predictions
    logger.log("validation_loss", avg_val_loss)
    logger.log("validation_accuracy", val_accuracy)
    logger.dump()

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Accuracy = {val_accuracy:.4f}")

    # Save the best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), logger.param_file)
        print(f"  New best model saved with validation accuracy: {best_val_accuracy:.4f}")

print("Behavioral Cloning training complete.")

save_as_onnx(model, sample_state, logger.onnx_file)

# Evaluate the agent in the real environment

### Environment and Agent

We provide some wrappers you need in order to get the same states from the environment as in the demonstration dataset.
Additionally the `RecordState` wrapper should be very helpful in collecting new samples for the DAgger algorithm.

In [ ]:
class CropObservation(gym.ObservationWrapper):
    def __init__(self, env, shape):
        gym.ObservationWrapper.__init__(self, env)
        self.shape = shape
        obs_shape = self.shape + env.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        return observation[:self.shape[0], :self.shape[1]]


class RecordState(gym.Wrapper):
    def __init__(self, env: gym.Env, reset_clean: bool = True):
        gym.Wrapper.__init__(self, env)

        assert env.render_mode is not None
        self.frame_list = []
        self.reset_clean = reset_clean

    def step(self, action, **kwargs):
        output = self.env.step(action, **kwargs)
        self.frame_list.append(output[0])
        return output

    def reset(self, *args, **kwargs):
        result = self.env.reset(*args, **kwargs)

        if self.reset_clean:
            self.frame_list = []
        self.frame_list.append(result[0])

        return result

    def render(self):
        frames = self.frame_list
        self.frame_list = []
        return frames


class Agent():
    def __init__(self, model, device):
        self.model = model
        self.device = device

    def select_action(self, state):
        with torch.no_grad():
            state = torch.Tensor(state).unsqueeze(0).to(device) / 255.0 # rescale
            logits = self.model(state)
            if type(logits) is tuple:
                logits = logits[0]
            probs = Categorical(logits=logits)
            return probs.sample().cpu().numpy()[0]


def make_env(seed, capture_video=True):
    env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
    env = gym.wrappers.RecordEpisodeStatistics(env)
    if capture_video:
        env = gym.wrappers.RecordVideo(env, logger.video_dir)

    env = CropObservation(env, (84, 96))
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = RecordState(env, reset_clean=True)
    env = gym.wrappers.FrameStackObservation(env, 4)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return env


def run_episode(agent, show_progress=True, capture_video=True, seed=None):
    env = make_env(seed=seed, capture_video=capture_video)
    state, _ = env.reset()
    score = 0
    done = False
    if show_progress:
        progress = tqdm(desc="Score: 0")

    while not done:
        action = agent.select_action(state[-1][np.newaxis, ...])
        state, reward, terminated, truncated, _ = env.step(action)
        score += reward
        done = terminated or truncated
        if show_progress:
            progress.update()
            progress.set_description("Score: {:.2f}".format(score))
    env.close()

    if show_progress:
        progress.close()
    if capture_video:
        show_video(logger.video_dir)

    return score


## Evaluate behavioral cloning agent

Let's see how the agent is doing in the real environment

In [ ]:
train_policy = Agent(model, device)
score = run_episode(train_policy, show_progress=True, capture_video=True);
print(f"Score: {score:.2f}")

Since we often have high variance when evaluating RL agents we should evaluate the agent multiple times to get a better feeling for its performance.

In [ ]:
train_policy = Agent(model, device)
n_eval_episodes = 10
scores = []
for i in tqdm(range(n_eval_episodes), desc="Episode"):
    scores.append(run_episode(train_policy, show_progress=False, capture_video=False))
    print("Score: %d" % scores[-1])
print("Mean Score: %.2f (Std: %.2f)" %(np.mean(scores), np.std(scores)))

# DAGGER

Now we can implement DAgger, you have downloaded a relatively well trained model you can use as an expert for this purpose.

Load expert model that is provided as ONNX file.

## Load the expert

In [ ]:
# Load expert
expert_model = ConvertModel(onnx.load("expert.onnx"))
expert_model = expert_model.to(device)
# Freeze expert weights
for p in expert_model.parameters():
    p.requires_grad = False

expert_policy = Agent(expert_model, device)

Next, you have to implement the DAgger algorithm (see slides for details). This function implements the core idea of DAgger:


1. Choose the policy with probability beta
2. Sample T-step trajectories using this policy
3. Label the gathered states with the expert

The aggregation and training part are already implemented.

In [ ]:
# inner loop of DAgger
def dagger(env, train_policy, expert_policy, dataset, beta=1.):

    ######################
    ### YOUR CODE HERE ###
    ######################

    # Implement DAgger algorithm here
    # 1) Choose a policy (sample according to beta)
    # 2) Sample T-step trajectory with the chosen policy
    #    (T can be an entire episode or a single state, think about what makes more sense here and implement it accordingly)
    # 3) Label the state (or states) with your expert if they come from your training policy

    #### Note ####
    # To get an action for the current state from your training policy or expert policy:
    # action = policy.select_action(state)
    #
    # Your training policy requires a single grayscale state while
    # the expert policy requires four stacked grayscale states
    # You can prepare your state for the policy like so:
    # Train policy:
    #      np.expand_dims(state[-1], 0)
    # Expert policy:
    #      state


    # Due to the RecordState wrapper you can get the states from the environment by calling
    # env.render()
    # Doing so will clear the list and the next time you call .render() will return the new states since the last call.
    # Note: be careful with the last state

    # Finally, add collected states and the actions the expert would execute in them to the dataset
    # dataset.append(states, actions)


    collected_states = []
    expert_actions = []

    current_stacked_state, _ = env.reset()
    done = False

    while not done:
        if random.random() < beta:

            action = expert_policy.select_action(current_stacked_state)
        else:

            action = train_policy.select_action(current_stacked_state[-1][np.newaxis, ...])


            collected_states.append(current_stacked_state[-1])
            expert_actions.append(expert_policy.select_action(current_stacked_state))


        current_stacked_state, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated


    if collected_states:
        dataset.append(np.array(collected_states), np.array(expert_actions))



Put everything together now.
1. Create new samples using the DAgger algorithm
2. Continue training your agent
3. Export your fully trained agent as an ONNX file

In [ ]:
# Specify the google drive mount here if you want to store logs and weights there (and set it up earlier)
logger = Logger("logdir_dagger")
print("Saving state to {}".format(logger.basepath))

# start environment
env = make_env(seed=42, capture_video=False)

# Training
######################
### YOUR CODE HERE ###
######################

In [ ]:
# DAgger parameters
DAgger_rounds = 10
epochs_per_dagger_round = 5
beta_schedule = np.linspace(1.0, 0.1, DAgger_rounds)



dagger_train_set = DemonstrationDataset("train")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

best_val_accuracy = -float('inf')

print("Starting DAgger training...")
for i_dagger_round in range(DAgger_rounds):
    current_beta = beta_schedule[i_dagger_round]
    print(f"\n--- DAgger Round {i_dagger_round + 1}/{DAgger_rounds} (beta={current_beta:.2f}) ---")


    print("Collecting new demonstrations...")

    train_policy_dagger = Agent(model, device)
    dagger(env, train_policy_dagger, expert_policy, dagger_train_set, beta=current_beta)
    print(f"Dataset size after round {i_dagger_round + 1}: {len(dagger_train_set)} samples")


    print(f"Retraining for {epochs_per_dagger_round} epochs...")
    dagger_train_loader = torch.utils.data.DataLoader(dagger_train_set, batch_size=batch_size, num_workers=2, shuffle=True, drop_last=False, pin_memory=True)

    for epoch in range(epochs_per_dagger_round):
        # Training
        model.train()
        train_loss_sum_dagger = 0
        for states, actions in dagger_train_loader:
            states, actions = states.to(device), actions.to(device)
            optimizer.zero_grad()
            logits = model(states)
            loss = criterion(logits, actions)
            loss.backward()
            optimizer.step()
            train_loss_sum_dagger += loss.item()
            logger.log("dagger_training_loss", loss.item())

        avg_train_loss_dagger = train_loss_sum_dagger / len(dagger_train_loader)

        # Validation (on original val set, not augmented)
        model.eval()
        val_loss_sum_dagger = 0
        correct_predictions_dagger = 0
        total_predictions_dagger = 0
        with torch.no_grad():
            for states, actions in val_loader:
                states, actions = states.to(device), actions.to(device)
                logits = model(states)
                loss = criterion(logits, actions)
                val_loss_sum_dagger += loss.item()

                _, predicted = torch.max(logits, 1)
                total_predictions_dagger += actions.size(0)
                correct_predictions_dagger += (predicted == actions).sum().item()

        avg_val_loss_dagger = val_loss_sum_dagger / len(val_loader)
        val_accuracy_dagger = correct_predictions_dagger / total_predictions_dagger
        logger.log("dagger_validation_loss", avg_val_loss_dagger)
        logger.log("dagger_validation_accuracy", val_accuracy_dagger)

        print(f"  Epoch {epoch+1}/{epochs_per_dagger_round}: Train Loss = {avg_train_loss_dagger:.4f}, Val Loss = {avg_val_loss_dagger:.4f}, Val Acc = {val_accuracy_dagger:.4f}")

        # Save best model after each epoch of fine-tuning within a DAgger round
        if val_accuracy_dagger > best_val_accuracy:
            best_val_accuracy = val_accuracy_dagger
            torch.save(model.state_dict(), logger.param_file)
            print(f"  New best model saved with validation accuracy: {best_val_accuracy:.4f}")

    logger.dump() # Dump logs after each DAgger round


print("\nDAgger training complete. Exporting final ONNX model...")

model.load_state_dict(torch.load(logger.param_file))
save_as_onnx(model, sample_state, logger.onnx_file)
print("Final model exported to ONNX.")

env.close()

In [ ]:
n_eval_episodes = 10
scores = []
for i in tqdm(range(n_eval_episodes), desc="Episode"):
    scores.append(run_episode(train_policy, show_progress=False, capture_video=False))
    print("Score: %d" % scores[-1])
print("Mean Score: %.2f (Std: %.2f)" %(np.mean(scores), np.std(scores)))